# مركز المعرفة الجامعي الذكي — عرض توضيحي كامل
### AI University Knowledge Hub — Full Capstone Demo

هذا الدفتر يشغّل المشروع بالكامل، خطوة بخطوة، مع طباعة **نتائج واضحة ومختصرة** بعد كل مرحلة، بدل سجلات المكتبات التقنية المعقدة (Kafka, Spark, Airflow, Great Expectations).

**المشروع:** Modern Data Engineering for AI Systems — SDAIA Academy

**المراحل الخمس:**
1. استقبال المستندات وفحصها (Kafka + Pydantic)
2. التخزين المنظم Bronze ← Silver ← Gold (Delta Lake)
3. نظام الأسئلة والأجوبة الذكي (RAG)
4. التنسيق التلقائي الكامل (Apache Airflow)
5. فحص الجودة النهائي وسجل التتبع (Great Expectations + OpenLineage)

> ملاحظة: تشغيل هذا الدفتر كاملًا يستغرق حوالي 10-15 دقيقة (معظمها لتحميل نماذج الذكاء الاصطناعي وتشغيل Spark).

In [ ]:
import os

if not os.path.exists('/content/ai-university-knowledge-hub'):
    !git clone -q https://github.com/LujainAldujain/University_Hub.git /content/ai-university-knowledge-hub

%cd /content/ai-university-knowledge-hub
print('تم تحميل المشروع بنجاح')

In [ ]:
AIRFLOW_VERSION = '3.3.0'
import sys
PYTHON_VERSION = f'{sys.version_info.major}.{sys.version_info.minor}'
CONSTRAINT_URL = f'https://raw.githubusercontent.com/apache/airflow/constraints-{AIRFLOW_VERSION}/constraints-{PYTHON_VERSION}.txt'

!pip install -q apache-airflow=={AIRFLOW_VERSION} --constraint {CONSTRAINT_URL}
!pip install -q -r requirements.txt
print('✅ تم تثبيت جميع الحزم المطلوبة')

In [ ]:
import logging

for noisy in ['kafka', 'py4j', 'great_expectations', 'openlineage', 'chromadb',
              'sentence_transformers', 'httpx', 'urllib3', 'airflow']:
    logging.getLogger(noisy).setLevel(logging.ERROR)

from loguru import logger as _loguru_logger
_loguru_logger.remove()

def section(title):
    print()
    print('=' * 60)
    print('  ' + title)
    print('=' * 60)

def ok(msg):
    print('✅ ' + msg)

def bad(msg):
    print('❌ ' + msg)

def info(msg):
    print('ℹ️  ' + msg)

print('تم تجهيز أدوات العرض الواضح')

In [ ]:
section('تجهيز البنية التحتية: تشغيل خادم Kafka محلي')

!curl -sSOL https://downloads.apache.org/kafka/4.2.1/kafka_2.13-4.2.1.tgz
!tar -xzf kafka_2.13-4.2.1.tgz

import subprocess
import time

kafka_dir = 'kafka_2.13-4.2.1'
cluster_id = subprocess.check_output([kafka_dir + '/bin/kafka-storage.sh', 'random-uuid']).decode().strip()
subprocess.run([kafka_dir + '/bin/kafka-storage.sh', 'format', '-t', cluster_id,
                '-c', kafka_dir + '/config/server.properties', '--standalone'], check=True)
subprocess.Popen([kafka_dir + '/bin/kafka-server-start.sh', kafka_dir + '/config/server.properties'],
                  stdout=open('/content/kafka.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(12)

for topic in ['university.documents.raw', 'university.documents.validated', 'university.documents.dlq']:
    subprocess.run([kafka_dir + '/bin/kafka-topics.sh', '--create', '--topic', topic,
                    '--bootstrap-server', 'localhost:9092', '--partitions', '3', '--replication-factor', '1'],
                   check=True, capture_output=True)

ok('خادم Kafka يعمل الآن، وتم إنشاء القنوات الثلاث (خام / مقبول / مرفوض)')

In [ ]:
import os

os.environ['AIRFLOW_HOME'] = '/content/ai-university-knowledge-hub/airflow'
os.environ['AIRFLOW__CORE__LOAD_EXAMPLES'] = 'False'

!airflow db migrate > /content/airflow_setup.log 2>&1
!python scripts/seed_sample_documents.py > /content/seed.log 2>&1

ok('تم تجهيز قاعدة بيانات Airflow وإضافة المستندات التجريبية')

## المرحلة 1: استقبال المستندات وفحصها
النظام يستقبل المستندات، ويتحقق من كل واحد منها (نوع الملف، حجمه، اكتمال بياناته) قبل قبوله.

In [ ]:
import sys
sys.path.insert(0, '/content/ai-university-knowledge-hub')

from producer.document_producer import main as produce
from consumer.document_consumer import main as consume

section('المرحلة 1: استقبال المستندات وفحصها')

produce()
result = consume()

ok('تم إرسال ' + str(result['total']) + ' مستندات')
ok(str(result['accepted']) + ' مقبولة')
bad(str(result['rejected']) + ' مرفوضة')
print()
info('أسباب الرفض:')
for reason, count in result['rejection_reasons'].items():
    print('   • ' + reason + '  (×' + str(count) + ')')

## المرحلة 2: التخزين المنظم (Bronze ← Silver ← Gold)
المستندات المقبولة تُنظّف وتُخزّن بشكل منظم عبر ثلاث طبقات: **خام (Bronze)** ← **منظّمة (Silver)** ← **ملخّصة (Gold)**، باستخدام Delta Lake.

In [ ]:
from lakehouse.bronze.bronze_loader import main as bronze_main
from lakehouse.silver.silver_transform import main as silver_main
from lakehouse.gold.gold_aggregate import main as gold_main

section('المرحلة 2: التخزين المنظم (Bronze ← Silver ← Gold)')

b = bronze_main()
ok('تم حفظ ' + str(b['row_count']) + ' مستندات في الطبقة الخام (Bronze)')

s = silver_main()
if s.get('revision_applied_this_run'):
    ok('تم تحديث مستند تمت مراجعته سابقًا في مكانه (لم يتكرر)')
if s.get('schema_enforcement_confirmed'):
    ok('النظام رفض تلقائيًا ملفًا حاول إضافة بيانات غير معروفة (حماية سلامة البيانات)')
ok('الطبقة المنظمة (Silver) تحتوي الآن ' + str(s['row_count']) + ' مستندات سليمة')

g = gold_main()
ok('تم تلخيص ' + str(g['silver_row_count']) + ' مستندات في ' + str(g['gold_row_count']) + ' فئات معرفية جاهزة للتحليل')

## المرحلة 3: نظام الأسئلة والأجوبة الذكي (RAG)
يجيب النظام على أسئلة الطلاب بالاعتماد **فقط** على المستندات الرسمية المخزّنة — ولا يخترع أي معلومة من عنده.

In [ ]:
from rag.rag_pipeline import RAGPipeline, EXAMPLE_QUESTIONS

section('المرحلة 3: نظام الأسئلة والأجوبة الذكي (RAG)')
info('جاري تجهيز الفهرسة الذكية (قد يستغرق دقيقة لتحميل نماذج الذكاء الاصطناعي)...')

pipeline = RAGPipeline()
ok('النظام جاهز للإجابة على الأسئلة')
print()

for q in EXAMPLE_QUESTIONS:
    result = pipeline.answer(q)
    print('❓ السؤال: ' + q)
    print('💬 الإجابة: ' + result['answer'])
    if result['citations']:
        print('📄 المصدر: ' + ', '.join(result['citations']))
    else:
        print('📄 المصدر: لا يوجد — النظام اعترف بعدم معرفته بدل اختراع إجابة')
    print('-' * 60)

## المرحلة 5: فحص الجودة النهائي وسجل التتبع
قبل اعتماد أي بيانات، يمر النظام على **6 فحوصات جودة صارمة** (Great Expectations). كما يسجّل النظام حدث بداية/نجاح/فشل لكل مرحلة (OpenLineage) — سجل تتبع كامل لكل عملية.

In [ ]:
from quality.ge_checkpoint import run_silver_quality_checkpoint
from lineage.lineage_emitter import emit_lineage

section('المرحلة 5: فحص الجودة النهائي (Great Expectations)')

with emit_lineage('quality_checkpoint_demo'):
    outcome = run_silver_quality_checkpoint()

for r in outcome['results']:
    label = '✅' if r['success'] else '❌'
    print(label + ' ' + r['expectation'] + '  (العمود: ' + str(r['column']) + ')')

print()
if outcome['success']:
    ok('جميع فحوصات الجودة الستة نجحت — البيانات جاهزة للاستخدام')
else:
    bad('فشل أحد فحوصات الجودة — يجب إيقاف المعالجة قبل المرحلة التالية')

### اختبار: ماذا يحدث إذا كانت البيانات معطوبة؟
لإثبات أن فحص الجودة **حقيقي** وليس مجرد شكلي، سنُدخل عمدًا خطأً حقيقيًا في البيانات (مستند مكرر) ونتأكد أن النظام يوقف المعالجة تلقائيًا.

In [ ]:
from deltalake import DeltaTable, write_deltalake
import pyarrow as pa

section('اختبار: حماية النظام من البيانات المعطوبة')

SILVER_PATH = '/content/ai-university-knowledge-hub/lakehouse/silver/documents_silver'
dt = DeltaTable(SILVER_PATH)
tbl = dt.to_pyarrow_table()
row = tbl.to_pylist()[0]
row['document_id'] = 'test-duplicate-0000'
write_deltalake(SILVER_PATH, pa.Table.from_pylist([row], schema=tbl.schema), mode='append')
info('تم إدخال مستند مكرر عمدًا لمحاكاة خطأ حقيقي (الملف: ' + row['file_name'] + ')')

gold_ran = False
try:
    with emit_lineage('quality_checkpoint_failure_demo'):
        result = run_silver_quality_checkpoint()
        if not result['success']:
            failed = [r['expectation'] for r in result['results'] if not r['success']]
            raise RuntimeError('فشل الفحص: ' + str(failed))
    gold_ran = True
    gold_main()
except RuntimeError:
    bad('تم إيقاف المعالجة تلقائيًا — لن يتم اعتماد البيانات الخاطئة')

print()
if not gold_ran:
    ok('النظام حمى نفسه بنجاح من البيانات المعطوبة')

dt = DeltaTable(SILVER_PATH)
dt.delete(predicate="document_id = 'test-duplicate-0000'")
ok('تمت إعادة البيانات إلى حالتها الصحيحة')

## المرحلة 4: التنسيق التلقائي الكامل (Apache Airflow)
كل ما سبق تم تشغيله يدويًا خطوة بخطوة لأغراض التوضيح. الآن سنُشغّل **نفس المشروع بالكامل تلقائيًا**، عبر أداة جدولة حقيقية تُستخدم فعليًا في الشركات الكبرى (Apache Airflow) — بضغطة واحدة، وبنفس ترتيب الاعتماديات، بحيث تتوقف أي مرحلة تالية تلقائيًا إذا فشلت مرحلة قبلها.

In [ ]:
import subprocess
from datetime import date

section('المرحلة 4: تشغيل خط الأنابيب بالكامل تلقائيًا عبر Airflow')
info('جاري التشغيل (قد يستغرق دقيقتين)...')

run_date = date.today().isoformat()
proc = subprocess.run(
    ['airflow', 'dags', 'test', 'university_knowledge_hub_pipeline', run_date],
    capture_output=True, text=True,
)
raw_output = proc.stdout + proc.stderr
with open('/content/airflow_dag_test_raw.log', 'w', encoding='utf-8') as f:
    f.write(raw_output)

TASK_LABELS = {
    'produce_documents': 'إرسال المستندات',
    'validate_and_consume': 'الفحص والتصنيف',
    'check_ingestion_quality': 'بوابة قبول الدفعة',
    'bronze_load': 'التخزين الخام (Bronze)',
    'silver_transform': 'التنظيف والتحديث (Silver)',
    'quality_checkpoint': 'فحص الجودة (Great Expectations)',
    'gold_aggregate': 'التلخيص المعرفي (Gold)',
}

task_names = []
returns = []
for line in raw_output.splitlines():
    if 'Current task name:' in line:
        task_names.append(line.split('Current task name:', 1)[1].strip())
    if 'Returned value was:' in line:
        returns.append(line.split('Returned value was:', 1)[1].strip())

run_success = 'state=success' in raw_output

print()
if task_names:
    for name in task_names:
        label = TASK_LABELS.get(name, name)
        ok(label)
else:
    info('لم يتم العثور على تفاصيل المهام في السجل — راجع الملف الكامل')

print()
if run_success:
    ok('اكتملت جميع مراحل المشروع تلقائيًا بنجاح عبر Airflow!')
else:
    bad('توقفت إحدى المراحل — راجع الملف الكامل: /content/airflow_dag_test_raw.log')

info('السجل التقني الكامل محفوظ في: /content/airflow_dag_test_raw.log (لمن يريد الاطلاع على التفاصيل التقنية)')

## ✅ الخلاصة
تم تشغيل **مركز المعرفة الجامعي الذكي** بالكامل بنجاح، من استقبال المستندات وحتى الإجابة على أسئلة الطلاب، مرورًا بجميع بوابات الجودة والتوثيق — يدويًا خطوة بخطوة، ثم تلقائيًا بالكامل عبر Airflow.

**النقطة الأهم:** النظام يجيب بدقة عند توفر المعلومة في المستندات الرسمية، ويعترف بعدم معرفته بدل اختراع إجابة عند غيابها.

مشروع تخرج — **Modern Data Engineering for AI Systems**, SDAIA Academy
الكود الكامل: https://github.com/LujainAldujain/University_Hub